In [ ]:
import pandas as pd
from langchain_community.document_loaders import CSVLoader # 안씀
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [77]:
# !pip install langchain-huggingface
# !pip install chromadb

In [91]:
df = pd.read_csv('./crawling/final/stay_crawling.csv')

documents = []

# 값이 없는 애들도 그냥 넣나..?
# 1. 컬럼값 여부에 상관없이 그냥 넣기
# 2. 컬럼값 없으면 해당 정보 담지 않음
def process_rows(row_name, row) -> str:
    content = row[row_name]
    if content is not None:
        return f"{row_name}: {content}"
    else:
        return ""
    
for _, row in df.iterrows():
    # 값 없는 애들은 nan
    content = f"""
    장소명: {row['장소명']}
    구분: {row['구분']}
    카테고리: {row['카테고리']}
    한줄설명: {row['한줄설명']}
    리뷰: {row['리뷰']}
    리뷰개수: {row['리뷰 개수']}
    평점: {row['평점']}
    태그: {row['태그']}
    가격: {row['가격']}
    영업시간: {row['영업시간']}
    주소: {row['주소']}
    연락처: {row['연락처']}
    출처: {row['출처']}
    """
    
    doc = Document(page_content=content.strip()) # 공백제거 하나용..? strip()
    documents.append(doc)

print(documents[5].page_content)

장소명: 라마다 앙코르 부산역 호텔
    구분: 숙박업체
    카테고리: 호텔
    한줄설명: 관광과 비즈니스의 중심지! 라마다앙코르부산역 호텔에서 편안한 휴식을 즐기세요.
    리뷰: ['안녕하세요. 25층에 조식 먹으러 갔다가 겪은 일입니다. 음식을 접시에 담고 있는데 어떤 투숙객이 조식 음식 세팅된 것을 동영상 찍으며 저의 모습도 담는 것 같아서 불편했습니다. 그래서 불편하다고 직원분께 말씀드렸더니 영상 삭제 조치를 취해주셨습니다. 저의 초상권을 지켜주셔서 감사합니다. 성실하고 적극적으로 불편사항 해소해준 직원분을 칭찬하고 싶습니다. 그리고 수건이 모자라서 추가요청하였는데 신속하고 뽀송뽀송한 제품 가져다 주셔서 좋았습니다. 또한 세면대 하부 별도 배수로가 없어서 물이 고이는 현상으로 미끄러워 넘어질수 있는 상태도 있어 조심조심 해야합니다. 나머지 부분은 매우 만족하고 재 방문의사가 있습니다. 감사합니다. ', '위치가 좋고 주변 편의시설도 많고 좋아요 ', '1018 체크인, 1019 체크아웃. 프론트가 왜 이렇게 4가지 없는지...  체크인 할때 남자직원은 예약자명 못찾길래 이름 헷갈려 하는거 같아서 OOO이 아니고 XXX이에요 라고 다시 정확히 얘기해주니깐 예약자명이 영문으로 되어있어서 못찾은거라고 기분 나쁘다는듯이 말대답하질 않나, 체크아웃 할때 여자직원은 소형 캐리어 하나 맡기고 나가려는데 얼마나 걸리냐고 신경질 부리질 않나..  분명 존댓말을 쓰고 있는데 어투는 신경질적이고 공격적으로 말하는게 너무 불쾌하다. 부산 사람들이 원래 이런건지? 부산이 우리나라 제 2의 도시이고 멋진 곳이라길래 기대하고 여행 왔는데 기분 다 잡치고 간다.  앞으로 부산 올 일 없을꺼고, 주변에도 부산 가지 말라고 할꺼다. 라마다는 직원 교육 및 관리 똑바로 해라. ']
    리뷰개수: 79개
    평점: 3.7
    태그: ['#라마다부산', '#부산역호텔', '#부산인피니티풀', '#라마다앙코르부산', '#부산역뷔페']
    가격: {'룸타입회원전용 [

In [102]:
# openAI key
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key를 입력하세요: ")

In [ ]:
# embedding..
# 모델 선정 이유... 그냥 써보고 싶어서..
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2") # 잘 안뽑아지는뎅..
# embeddings = HuggingFaceEmbeddings(model="intfloat/multilingual-e5-small") # 임베딩 크기 384


In [ ]:
# 문서에서 벡터 비디 생성
vector_store = Chroma.from_documents( # 문서리스트로부터 벡터 저장소 생성
    documents=documents, # 문서 리스트
    embedding=embeddings, # 임베딩 함수
    collection_name="stay", # 생성할 컬렉션 이름
    persist_directory="./chroma_db", # 컬렉션 저장할 디렉토리 -> 지정하지 않으면 메모리에 임시 저장
    # ids: None, # 문서 ID 리스트 지정안하면 UUID
    )

In [112]:
# 만들어진 db 에 연결 (load)
persist_db = Chroma(
    persist_directory="./chroma_db",
    embedding_function=OpenAIEmbeddings(),
    collection_name="stay", # 컬렉션 이름 같아야함
)

In [ ]:
# DB 에 값 추가
persist_db.add_texts(
    texts = ["해운대의 더러운 호텔로는 네이버호텔이 있습니다."],
    # metadatas = [{}]
    ids=["1"],
    embe
)

['1']

In [133]:
persist_db.get(["1"]) # id 로만 조회 가능 PK 같은건가봄?
persist_db.get(limit=5)

{'ids': ['4ea4f8ad-6bc4-4e9b-96b5-18df5f991869',
  'd8754d9d-bf99-4fb7-a796-e57db99ce39f',
  '669102a0-7685-46f2-b469-42f9f973a460',
  'afa2698f-2c7d-4088-884b-b776c44e3fbc',
  'fbbcc117-220d-424b-8fa0-2c6a1e085fae'],
 'embeddings': None,
 'documents': ["장소명: 롯데호텔 부산\n    구분: 숙박업체\n    카테고리: 호텔\n    한줄설명: nan\n    리뷰: ['롯데호텔 알바한사람입니다  저도 일을 잘 한 편은 아니지만 , 정말 잘못된 체계라고 생각합니다 . 정직원을 뽑으시던지 계약직을 쓰시던지 알바를 시키면서 계약직을 원하시면 어떻게하자는건지 모르겠습니다 일을 알려주지도 않고 알아서 다 하기를 바라는 신기한 상황을 처음보아 후기를 남깁니다.  알바생을 진짜 직원인것처럼 부려 먹는 신기한 연회장,호텔입니다  ', '연회했는데 직원들이 센스가 하나도 없어서, 쓰레기통 가져다달라, 뜨거운물 가져다달라, 하나하나 말을 해야 겨우 챙겨주고, 서비스는 별1개도 아깝습니다. ', '서면 중심부에 있어서 교통도 편리하고 근처에 맛집도 많아서 좋아요. 호텔은 오래되었지만 관리가 잘되어 있어서 괜찮았어요. 저녁에 실외 수영장도 라이트업 되어서 예뻤고 사진찍기 좋아요. 실내 수영장은 수영하기 좋구요. 라운지 음식도 맛있고 라세느 조식도 맛있어요. 또, 직원분들이 진짜 친절하셔서 좋았어요.    ']\n    리뷰개수: 116개\n    평점: 4.3\n    태그: ['#2인실', '#가족실', '#물놀이', '#비즈니스호텔', '#사우나', '#실외수영장', '#연회장', '#이동약자접근', '#조식서비스', '#호캉스']\n    가격: {'룸타입슈페리어 더블 (Superior Double)': '205,700원', '룸타입슈페리어 트윈 (Superior twin)'

In [134]:
# lang chain 썼을때...
query = "더러운 해운대의 호텔 추천해줘"
results = persist_db.similarity_search(query, k=3)

for res in results:
    print("-" * 30)
    print(res.page_content) 

------------------------------
해운대의 더러운 호텔로는 네이버호텔이 있습니다.
------------------------------
장소명: 호텔일루아
    구분: 숙박업체
    카테고리: 호텔
    한줄설명: nan
    리뷰: ['시설은 그냥 그랬고 뷰가 미텼습니다. 근데 들어오자마자 냄새가 나서 조금 그랬어요. 방은 좁지 않아어요. ', '뷰는 좋지만 시설이 너무 낡았다. 나머지는 다 괜찮은데 에어컨이 큰 문제였다. 새벽에 더워서 틀었는데 너무 시끄러워서 이어플러그 아니면 못 잘 뻔했다. 에어컨 소음만 보면 뉴욕 미드타운에 있는 호텔들 버금감 프론트에 말씀드렸더니 풀부킹이라 객실 변경이 불가하다고 하심. 레노베이션을 하든지 하자가 있는 객실은 판매를 안하든지 하셔야 할듯 ', '안개가 껴있어서 거의 보이지도않았지만..야경을 보고싶어서 예약한 시티뷰가 벽뷰였다..  시설이나 위치는 괜찮았다 직원분들도 친절했다.  ']
    리뷰개수: 43개
    평점: 4.4
    태그: ['#2인실', '#가족실', '#부티크호텔', '#산책로', '#스파룸', '#야외테라스', '#연회장', '#오션뷰', '#온돌룸', '#전망좋은호텔', '#조식서비스', '#호캉스']
    가격: {}
    영업시간: nan
    주소: 부산 해운대구 달맞이길 97
    연락처: 051-744-1331
    출처: 카카오맵(2025)
------------------------------
장소명: 호텔일루아
    구분: 숙박업체
    카테고리: 호텔
    한줄설명: nan
    리뷰: ['시설은 그냥 그랬고 뷰가 미텼습니다. 근데 들어오자마자 냄새가 나서 조금 그랬어요. 방은 좁지 않아어요. ', '뷰는 좋지만 시설이 너무 낡았다. 나머지는 다 괜찮은데 에어컨이 큰 문제였다. 새벽에 더워서 틀었는데 너무 시끄러워서 이어플러그 아니면 못 잘 뻔했다. 에어컨 소음만 보면 뉴욕 미드타운에 있는 호텔들 버금감 프론트에 말씀드렸더니 풀부

In [128]:
# retriever = persist_db.as_retriever( # retriever 
#     search_type="similarity", # 유사도 검색
#     search_kwargs={"k": 3}    # 3 개 뽑기
# )

SEARCH_TYPE = [
    { # mmr 전용
    "k": 3, 
    "fetch_k": 10, 
    "lambda_mult": 0.5 # 다양성 조절 -> 0 으로 갈수록 다양성 up 
    },
    { 
    "k": 5,
    "score_threshold": 0.8 # 유사도 점수가 기준 이상인 애들만
    },
    {
    "k": 5,
    "distance_threshold": 0.2 # 임베딩 거리 기준
    },
    {
    "k": 3,
    "filter": {"source": "카카오맵(2025)"} # 메타 데이터 필터링!
    },]

# 상관없는 문서 없애기
# retriever = persist_db.as_retriever(
#     search_type="similarity_score_threshold",
#     search_kwargs=SEARCH_TYPE[1]
# )

# mmr(유사도 + 다양성) 전용값들
retriever = persist_db.as_retriever(
    search_type="mmr",
    search_kwargs= SEARCH_TYPE[0]
)

llm = ChatOpenAI(
    model="gpt-4o", 
    temperature=0
    )

template = """
당신은 장소 추천 전문가로서 여행 계획 플래너가 여행 계획을 세울 때 참고할 수 있는 숙박 업소를 추출하는 역할을 맡았습니다.
아래 [정보]를 바탕으로 질문에 친절하게 답해주세요. 안 좋은 정보라도 뽑아서 답하세요.
[정보]에 없는 내용은 지어내지 말고 "정보가 없습니다"라고 답하세요.

[정보]:
{context}

질문: {question}
"""
prompt = ChatPromptTemplate.from_template(template) # langchain 용 프롬프트 객체

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs]) # document -> str

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

query = "더러운 해운대 호텔 3개 추천해줄 수 있어? 참고해서 거기는 예약안하게"
response = rag_chain.invoke(query)

print(f"질문: {query}")
print(f"답변:\n{response}")


질문: 더러운 해운대 호텔 3개 추천해줄 수 있어? 참고해서 거기는 예약안하게
답변:
[정보]에 제공된 호텔들 중에서 "더럽다"는 직접적인 언급은 없습니다. 그러나, 호텔일루아의 경우 "들어오자마자 냄새가 나서 조금 그랬어요"라는 리뷰가 있어 청결에 대한 우려가 있을 수 있습니다. 다른 두 호텔인 오션더포인트호텔과 더클럽호텔 연화리점에 대해서는 청결과 관련된 부정적인 리뷰가 없습니다. 따라서, 더러운 호텔을 추천해드리기는 어렵습니다. 추가적인 정보가 필요하다면 다른 출처를 참고하시기 바랍니다.
